In [1]:
from pathlib import Path
import pandas as pd


## Traficom M1 preprocessing
This notebook preprocesses Traficom M1 passenger car registry data and builds cleaned brand-level and model-level summary tables for later merging with the spare-parts pricing dataset.

In [ ]:
path = "/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/traficom_datasets/TieliikenneAvoinData_31_12_2025.csv"
df = pd.read_csv(path, sep=";", encoding="latin1", low_memory=False)

## Load libraries and data
Read the Traficom extract and keep the notebook focused on the fields needed for final market-structure summary tables.

## Inspect raw columns and sample values
Review the retained Traficom fields, missingness, and representative coded values before cleaning.

## Notebook workflow
1. Load libraries and data
2. Inspect raw columns and sample values
3. Clean and standardize fields
4. Parse dates and derive year and age features
5. Filter to Finnish passenger car market scope used in this thesis
6. Standardize brand and model keys for aggregation
7. Build `brand_summary`
8. Build `model_summary`
9. Inspect target brands and models
10. Export final summary tables

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.info())

(5122260, 41)
['ajoneuvoluokka', 'ensirekisterointipvm', 'ajoneuvoryhma', 'ajoneuvonkaytto', 'variantti', 'versio', 'kayttoonottopvm', 'vari', 'ovienLukumaara', 'korityyppi', 'ohjaamotyyppi', 'istumapaikkojenLkm', 'omamassa', 'teknSuurSallKokmassa', 'tieliikSuurSallKokmassa', 'ajonKokPituus', 'ajonLeveys', 'ajonKorkeus', 'kayttovoima', 'iskutilavuus', 'suurinNettoteho', 'sylintereidenLkm', 'ahdin', 'sahkohybridi', 'sahkohybridinluokka', 'merkkiSelvakielinen', 'mallimerkinta', 'vaihteisto', 'vaihteidenLkm', 'kaupallinenNimi', 'voimanvalJaTehostamistapa', 'tyyppihyvaksyntanro', 'yksittaisKayttovoima', 'kunta', 'NEDC_Co2', 'NEDC2_Co2', 'WLTP_Co2', 'WLTP2_Co2', 'matkamittarilukema', 'valmistenumero2', 'jarnro']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5122260 entries, 0 to 5122259
Data columns (total 41 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   ajoneuvoluokka             object 
 1   ensirekisterointipvm       object 
 2   ajone

In [ ]:
cols_keep = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "variantti",
    "versio",
    "ensirekisterointipvm",
    "kayttoonottopvm",
    "ajoneuvoluokka",
    "ajoneuvoryhma",
    "korityyppi",
    "kayttovoima",
    "yksittaisKayttovoima",
    "sahkohybridi",
    "sahkohybridinluokka",
    "iskutilavuus",
    "suurinNettoteho",
    "sylintereidenLkm",
    "omamassa",
    "vaihteisto",
    "vaihteidenLkm",
    "ahdin",
    "ovienLukumaara",
    "istumapaikkojenLkm",
    "voimanvalJaTehostamistapa",
    "matkamittarilukema"
]

traficom_reduced = df[cols_keep].copy()

print(traficom_reduced.shape)
traficom_reduced.head()

# Keep the reduced table in memory for now; no exports yet.


(5122260, 25)


,merkkiSelvakielinen,mallimerkinta,kaupallinenNimi,variantti,versio,ensirekisterointipvm,kayttoonottopvm,ajoneuvoluokka,ajoneuvoryhma,korityyppi,...,suurinNettoteho,sylintereidenLkm,omamassa,vaihteisto,vaihteidenLkm,ahdin,ovienLukumaara,istumapaikkojenLkm,voimanvalJaTehostamistapa,matkamittarilukema
0,BMW,R60/590,NaN,NaN,NaN,NaN,19670000,MUU,"21, 512",NaN,...,NaN,NaN,210.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN
1,Sprite,ALPINE/C,NaN,NaN,NaN,01.09.1976,19760000,MUU,"9, 13, 513",NaN,...,NaN,NaN,630.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
2,Omavalmiste,PV350/2000,NaN,NaN,NaN,22.09.1983,19830000,MUU,"13, 20, 513",NaN,...,NaN,NaN,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Valtteri,LKZ-8101,NaN,NaN,NaN,09.02.1994,19940209,O1,"1, 13",NaN,...,NaN,NaN,170.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Honda,4D ACCORD SEDAN 2.0-CL75/268,ACCORD,1,4,02.10.2003,20031002,M1,NaN,AA,...,114.0,4.0,1462.0,NaN,NaN,NaN,NaN,5.0,5.0,293108.0


In [ ]:
missing = (
    traficom_reduced.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
)

print(missing)

sahkohybridinluokka          0.924085
vaihteidenLkm                0.588855
sahkohybridi                 0.570312
ajoneuvoryhma                0.542370
ovienLukumaara               0.520365
vaihteisto                   0.486266
ahdin                        0.474071
matkamittarilukema           0.448032
variantti                    0.374765
korityyppi                   0.359669
versio                       0.339221
sylintereidenLkm             0.331299
suurinNettoteho              0.330044
iskutilavuus                 0.265691
kaupallinenNimi              0.231678
istumapaikkojenLkm           0.231567
yksittaisKayttovoima         0.230334
kayttovoima                  0.230329
voimanvalJaTehostamistapa    0.195374
mallimerkinta                0.178238
ensirekisterointipvm         0.030804
omamassa                     0.002126
merkkiSelvakielinen          0.000360
ajoneuvoluokka               0.000000
kayttoonottopvm              0.000000
Name: missing_share, dtype: float64


In [ ]:
cat_cols = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "ajoneuvoluokka",
    "ajoneuvoryhma",
    "korityyppi",
    "kayttovoima",
    "yksittaisKayttovoima",
    "sahkohybridi",
    "sahkohybridinluokka",
    "vaihteisto",
    "ahdin",
]

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(traficom_reduced[col].value_counts(dropna=False).head(15))


--- merkkiSelvakielinen ---
merkkiSelvakielinen
Toyota            442877
Volvo             295934
Ford              264593
Mercedes-Benz     263777
Volkswagen, VW    189455
Muuli             180017
Skoda             174880
Volkswagen        172155
BMW               147263
Nissan            143570
Omavalmiste       129396
Audi              125486
Kia               118342
Opel              110207
Honda              92451
Name: count, dtype: int64

--- mallimerkinta ---
mallimerkinta
NaN                                                   912979
TOYOTA COROLLA Farmari (AC) 4ov 1798cm3                17589
XC60 Farmari (AC) 5ov 1969cm3 A                        16893
1250 XI/250                                            16596
TRANSPORTER Umpikorinen (BB) 6ov 1968cm3               16537
TOYOTA RAV4 Farmari (AC) 4ov 2487cm3                   15669
TOYOTA AURIS Monikäyttöajoneuvo (AF) 4ov 1798cm3       15447
Nissan Qashqai Monikäyttöajoneuvo (AF) 4ov 1197cm3     14527
TOYOTA YARIS Viistoperä (

## Clean and standardize fields
Parse raw fields into usable text, date, and numeric formats before deriving summary features.

In [ ]:
df = traficom_reduced.copy()

# Traficom uses two different raw date formats here:
# ensirekisterointipvm is typically dd.mm.yyyy, while kayttoonottopvm is yyyymmdd.
# The previous generic to_datetime call parsed yyyymmdd integers as Unix ns timestamps,
# which collapsed use_year to 1970 and inflated vehicle_age.
date_cols = ["ensirekisterointipvm", "kayttoonottopvm"]
df["ensirekisterointipvm"] = pd.to_datetime(
    df["ensirekisterointipvm"].astype("string").str.strip(),
    format="%d.%m.%Y",
    errors="coerce",
)

raw_use_date = df["kayttoonottopvm"].astype("string").str.strip()
raw_use_date = raw_use_date.str.replace(r"\\.0$", "", regex=True)
raw_use_date = raw_use_date.where(raw_use_date.str.len() == 8, raw_use_date.str.zfill(8))
raw_use_date = raw_use_date.where(~raw_use_date.str.match(r"^\\d{4}0000$"), raw_use_date.str[:4] + "0101")
df["kayttoonottopvm"] = pd.to_datetime(raw_use_date, format="%Y%m%d", errors="coerce")

num_cols = [
    "iskutilavuus",
    "suurinNettoteho",
    "sylintereidenLkm",
    "omamassa",
    "vaihteidenLkm",
    "ovienLukumaara",
    "istumapaikkojenLkm",
    "matkamittarilukema",
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df[["ensirekisterointipvm", "kayttoonottopvm"] + num_cols].info()
df[num_cols].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5122260 entries, 0 to 5122259
Data columns (total 10 columns):
 #   Column                Dtype         
---  ------                -----         
 0   ensirekisterointipvm  datetime64[ns]
 1   kayttoonottopvm       datetime64[ns]
 2   iskutilavuus          float64       
 3   suurinNettoteho       float64       
 4   sylintereidenLkm      float64       
 5   omamassa              float64       
 6   vaihteidenLkm         float64       
 7   ovienLukumaara        float64       
 8   istumapaikkojenLkm    float64       
 9   matkamittarilukema    float64       
dtypes: datetime64[ns](2), float64(8)
memory usage: 390.8 MB


,iskutilavuus,suurinNettoteho,sylintereidenLkm,omamassa,vaihteidenLkm,ovienLukumaara,istumapaikkojenLkm,matkamittarilukema
count,3.761320e+06,3.431689e+06,3.425262e+06,5.111370e+06,2.105992e+06,2.456816e+06,3.936116e+06,2.827326e+06
mean,2.124861e+03,9.822774e+01,3.873830e+00,1.664812e+03,5.832753e+00,4.211170e+00,4.141060e+00,2.034901e+05
std,6.532973e+03,5.443175e+01,4.325418e+00,2.579645e+03,3.122946e+00,8.452170e-01,2.647737e+00,5.782477e+05
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,1.390000e+03,7.200000e+01,4.000000e+00,4.400000e+02,5.000000e+00,4.000000e+00,3.000000e+00,1.009550e+05
50%,1.798000e+03,9.190000e+01,4.000000e+00,1.375000e+03,6.000000e+00,4.000000e+00,5.000000e+00,1.805235e+05
75%,2.198000e+03,1.150000e+02,4.000000e+00,1.800000e+03,7.000000e+00,5.000000e+00,5.000000e+00,2.739440e+05
max,6.700000e+06,1.663500e+04,6.138000e+03,9.539560e+05,3.636000e+03,5.600000e+01,1.030000e+02,5.407337e+08


In [ ]:
text_cols = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "variantti",
    "versio",
]

for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

In [ ]:
df["brand"] = df["merkkiSelvakielinen"]
df["model"] = df["mallimerkinta"]

In [ ]:
print(df["brand"].value_counts(dropna=False).head(20))
print(df["model"].value_counts(dropna=False).head(20))

brand
toyota            442880
volvo             295934
ford              264593
mercedes-benz     263777
volkswagen, vw    189455
muuli             180017
skoda             174880
volkswagen        172158
bmw               147263
nissan            143570
omavalmiste       129513
audi              125486
kia               118344
opel              110207
honda              92453
peugeot            87760
aku                86478
respo              78208
valmet             71355
majava             68350
Name: count, dtype: Int64
model
<NA>                                                  912979
toyota corolla farmari (ac) 4ov 1798cm3                17592
xc60 farmari (ac) 5ov 1969cm3 a                        16895
nissan qashqai monikäyttöajoneuvo (af) 4ov 1197cm3     16781
1250 xi/250                                            16596
transporter umpikorinen (bb) 6ov 1968cm3               16559
toyota rav4 farmari (ac) 4ov 2487cm3                   15680
toyota auris monikäyttöajoneuvo (af

In [ ]:
for col in ["ajoneuvoluokka", "ajoneuvoryhma"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(30))


--- ajoneuvoluokka ---
ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
O4       54939
N1G      54728
L1e      53280
MTK      50535
T        49790
N3       46526
T1       45547
L1       32558
N2       30763
LTR      11468
M3        8596
N2G       7458
T2        6591
L7e       4493
L6e       3546
N3G       2922
L2e       2282
M2        2263
L2        1320
L4         726
Name: count, dtype: int64

--- ajoneuvoryhma ---
ajoneuvoryhma
NaN            2778158
1, 13           603755
74              183494
75, 509         156317
2, 13           145061
109             138175
75               97355
509              94253
13, 20, 513      78761
6, 13            65029
107              62398
75, 933          49134
9, 13            47708
21               44689
7, 13            41542
73               33980
14               26529
13               24920
13, 20           24540
85         

## Filter to M1 passenger cars
Restrict the cleaned registry extract to the Finnish passenger car market scope used in this thesis: Traficom class `M1`.

In [ ]:
vehicle_class_counts = df["ajoneuvoluokka"].astype("string").str.strip().value_counts(dropna=False)
print(vehicle_class_counts.head(10))

m1_df = df[df["ajoneuvoluokka"].astype("string").str.strip().str.lower().eq("m1")].copy()
print(m1_df.shape)

ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
Name: count, dtype: Int64
(2565062, 27)


In [ ]:
for col in ["mallimerkinta", "kaupallinenNimi", "variantti", "versio"]:
    print(f"\n--- {col} ---")
    print(m1_df[col].value_counts(dropna=False).head(20))


--- mallimerkinta ---
mallimerkinta
toyota corolla farmari (ac) 4ov 1798cm3                    17552
nissan qashqai monikäyttöajoneuvo (af) 4ov 1197cm3         16779
toyota rav4 farmari (ac) 4ov 2487cm3                       15637
toyota auris monikäyttöajoneuvo (af) 4ov 1798cm3           15455
toyota yaris viistoperä (ab) 4ov 1490cm3                   14348
model y monikäyttöajoneuvo (af) 5ov                        14207
model 3 sedan (aa) 4ov                                     13802
toyota yaris monikäyttöajoneuvo (af) 4ov 1329cm3           13745
toyota c-hr viistoperä (ab) 4ov 1798cm3                    13324
v60 farmari (ac) 5ov 1969cm3 a                             11517
nissan qashqai monikäyttöajoneuvo (af) 4ov 1598cm3         11409
toyota avensis monikäyttöajoneuvo (af) 4ov 1798cm3         11269
fiesta viistoperä (ab) 4ov 998cm3                          10404
passat farmari (ac) 5ov 1395cm3 a                          10250
toyota yaris hybrid monikäyttöajoneuvo (af) 4ov 1497c

In [ ]:
m1_df.loc[~m1_df["iskutilavuus"].between(500, 10000, inclusive="both"), "iskutilavuus"] = pd.NA
m1_df.loc[~m1_df["suurinNettoteho"].between(20, 1000, inclusive="both"), "suurinNettoteho"] = pd.NA
m1_df.loc[~m1_df["sylintereidenLkm"].between(2, 16, inclusive="both"), "sylintereidenLkm"] = pd.NA
m1_df.loc[~m1_df["omamassa"].between(500, 5000, inclusive="both"), "omamassa"] = pd.NA
m1_df.loc[~m1_df["matkamittarilukema"].between(0, 1000000, inclusive="both"), "matkamittarilukema"] = pd.NA
m1_df.loc[~m1_df["ovienLukumaara"].between(2, 6, inclusive="both"), "ovienLukumaara"] = pd.NA
m1_df.loc[~m1_df["istumapaikkojenLkm"].between(1, 9, inclusive="both"), "istumapaikkojenLkm"] = pd.NA

In [ ]:
m1_df["model_family"] = (
    m1_df["kaupallinenNimi"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(m1_df["model_family"].value_counts(dropna=False).head(30))

model_family
golf              86763
octavia           82555
corolla           59454
focus             57105
<NA>              52363
passat            50893
nissan qashqai    47914
v70               43959
avensis           40308
toyota corolla    39081
toyota yaris      35843
ceed              35509
fiesta            31895
polo              31582
toyota auris      29253
toyota avensis    26980
toyota rav4       26596
mondeo            25838
v60               23427
superb            20840
yaris             20157
rio               20086
fabia             18620
s60               17703
toyota c-hr       16627
clio              15937
a4                15544
i20               14531
almera            14227
model y           14219
Name: count, dtype: Int64


## Parse dates and derive year and age features
Construct a stable vehicle-year proxy and derive age bands used in the final market summaries.

In [ ]:
registration_year = m1_df["ensirekisterointipvm"].dt.year
use_year = m1_df["kayttoonottopvm"].dt.year
vehicle_year = use_year.fillna(registration_year)

m1_df["registration_year"] = registration_year
m1_df["use_year"] = use_year
m1_df["vehicle_year"] = vehicle_year

date_feature_check = m1_df[["ensirekisterointipvm", "kayttoonottopvm", "registration_year", "use_year", "vehicle_year"]].head(10)
date_feature_check

,ensirekisterointipvm,kayttoonottopvm,registration_year,use_year,vehicle_year
4,2003-10-02,2003-10-02,2003.0,2003.0,2003.0
5,2006-03-17,2006-03-17,2006.0,2006.0,2006.0
6,2007-01-05,2007-01-05,2007.0,2007.0,2007.0
8,1996-03-14,1996-03-14,1996.0,1996.0,1996.0
10,2000-03-24,2000-03-24,2000.0,2000.0,2000.0
12,NaT,NaT,NaN,NaN,NaN
19,1999-06-21,1999-06-21,1999.0,1999.0,1999.0
22,1993-11-16,1993-11-16,1993.0,1993.0,1993.0
31,2006-01-10,2006-01-10,2006.0,2006.0,2006.0
32,2008-01-22,2008-01-22,2008.0,2008.0,2008.0


In [ ]:
reference_year = 2025
m1_df.loc[~m1_df["vehicle_year"].between(1950, 2025, inclusive="both"), "vehicle_year"] = pd.NA
m1_df["vehicle_age"] = reference_year - m1_df["vehicle_year"]
m1_df.loc[~m1_df["vehicle_age"].between(0, 80, inclusive="both"), "vehicle_age"] = pd.NA

m1_df[["vehicle_year", "vehicle_age"]].describe()

,vehicle_year,vehicle_age
count,2.557839e+06,2.557839e+06
mean,2.012652e+03,1.234808e+01
std,7.983035e+00,7.983035e+00
min,1.950000e+03,0.000000e+00
25%,2.007000e+03,6.000000e+00
50%,2.014000e+03,1.100000e+01
75%,2.019000e+03,1.800000e+01
max,2.025000e+03,7.500000e+01


In [ ]:
def make_year_band(y):
    if pd.isna(y):
        return pd.NA
    y = int(y)
    if y < 2005:
        return "pre_2005"
    elif y <= 2009:
        return "2005_2009"
    elif y <= 2014:
        return "2010_2014"
    elif y <= 2019:
        return "2015_2019"
    else:
        return "2020_2025"

m1_df["year_band"] = m1_df["vehicle_year"].apply(make_year_band)

## Standardize grouping keys
Create stable brand and model-family aggregation keys before building the final market-structure summary tables.

In [ ]:
# Standardize brand labels for grouping.
m1_df["brand_clean"] = (
    m1_df["brand"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
    "vw": "volkswagen",
    "volkswagen, vw": "volkswagen",
    "skd": "skoda",
    "skida": "skoda",
    })
)

# Standardize model-family labels for grouping.
m1_df["model_family_clean"] = (
    m1_df["model_family"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\\s+", " ", regex=True)
    .str.replace(r"^(toyota|volkswagen|vw|skoda)[\\s-]+", "", regex=True)
)

In [ ]:
# Check that obvious duplicate brand labels collapse into one cleaned key.
brand_key_check = (
    m1_df.loc[
        m1_df["brand"].astype("string").str.contains("vw|volkswagen|skd|skida|skoda", case=False, na=False),
        ["brand", "brand_clean"],
    ]
    .drop_duplicates()
    .sort_values(["brand_clean", "brand"])
)

brand_key_check.head(20)

# Check that brand-prefixed model labels collapse into one cleaned model family.
model_key_check = (
    m1_df.loc[
        m1_df["model_family"].astype("string").str.contains("corolla|octavia|golf", case=False, na=False),
        ["brand_clean", "model_family", "model_family_clean"],
    ]
    .drop_duplicates()
    .sort_values(["brand_clean", "model_family_clean", "model_family"])
)

model_key_check.head(30)

,brand_clean,model_family,model_family_clean
4094647,skoda,2d octavia super,2d octavia super
4426210,skoda,2d octavia super/2400,2d octavia super/2400
607769,skoda,4d octavia combi 1.6-75-a-1z/258,4d octavia combi 1.6-75-a-1z/258
1899109,skoda,4d octavia combi 1.9tdi-77-s-1z/258,4d octavia combi 1.9tdi-77-s-1z/258
2146648,skoda,4d octavia hatchback 1.6-1u/250,4d octavia hatchback 1.6-1u/250
1727112,skoda,4d octavia hatchback 1.6-75-x-1u/250,4d octavia hatchback 1.6-75-x-1u/250
2303073,skoda,4d octavia rs combi 2.0tdi dpf-125-h-1z/258,4d octavia rs combi 2.0tdi dpf-125-h-1z/258
1902998,skoda,4d octavia rs sedan 2.0tdi dpf-125-h-1z/2580,4d octavia rs sedan 2.0tdi dpf-125-h-1z/2580
3648850,skoda,4d octavia sedan 1.6fsi-85-b-1z/258,4d octavia sedan 1.6fsi-85-b-1z/258
35282,skoda,4d octavia sedan 1.9tdi-77-s-1z/258,4d octavia sedan 1.9tdi-77-s-1z/258


## Final Traficom summary outputs
`brand_summary` is the brand-level market structure table and `model_summary` is the model-family-level market structure table. These are the final Traficom outputs from this notebook and will later be merged with the spare-parts pricing dataset.

In [ ]:
# Review the dominant coded values used in summary features before mapping them.
for col in ["vaihteisto", "kayttovoima", "sahkohybridi", "ahdin"]:
    print(f"\\n--- {col} ---")
    print(m1_df[col].astype("string").value_counts(dropna=False).head(12))

\n--- vaihteisto ---
vaihteisto
2       902472
1       705119
<NA>    622373
3       224562
8        50093
Y        49441
X         7833
9         1919
6          857
4          328
5           29
7           26
Name: count, dtype: Int64
\n--- kayttovoima ---
kayttovoima
01      1805932
02       572766
04       164607
38         8148
13         8124
40         4420
39          891
44           58
<NA>         50
05           17
42           10
34            9
Name: count, dtype: Int64
\n--- sahkohybridi ---
sahkohybridi
False    1186446
<NA>     1033205
True      345411
Name: count, dtype: Int64
\n--- ahdin ---
ahdin
True     1270912
False     683870
<NA>      610280
Name: count, dtype: Int64


In [ ]:
# Verified from the current Traficom extract value counts above.
petrol_codes = {"01"}
diesel_codes = {"02"}
ev_codes = {"04"}

# sahkohybridi and ahdin are stored as string booleans in this extract.
true_values = {"True"}

# Transmission code handling should be validated against the official Traficom codebook
# before being used for external reporting. In this notebook, codes 2 and Y are treated
# as automatic-like transmissions based on the observed M1 values in the current extract.
automatic_codes = {"2", "Y"}

total_m1 = len(m1_df)

## Build `brand_summary`
Aggregate the cleaned M1 market to brand level and keep the final output focused on reusable merge features.

In [ ]:
# Build the final brand-level summary using the cleaned brand key.
brand_summary_aggs = {
    "brand_total_registered": ("brand_clean", "size"),
}

if "vehicle_age" in m1_df.columns:
    brand_summary_aggs["brand_median_vehicle_age"] = ("vehicle_age", "median")
    brand_summary_aggs["brand_mean_vehicle_age"] = ("vehicle_age", "mean")

if "matkamittarilukema" in m1_df.columns:
    brand_summary_aggs["brand_median_mileage"] = ("matkamittarilukema", "median")
    brand_summary_aggs["brand_mean_mileage"] = ("matkamittarilukema", "mean")

if "iskutilavuus" in m1_df.columns:
    brand_summary_aggs["brand_median_engine_cc"] = ("iskutilavuus", "median")

if "suurinNettoteho" in m1_df.columns:
    brand_summary_aggs["brand_median_power_kw"] = ("suurinNettoteho", "median")

if "omamassa" in m1_df.columns:
    brand_summary_aggs["brand_median_mass_kg"] = ("omamassa", "median")

brand_summary = (
    m1_df.groupby("brand_clean", dropna=False)
    .agg(**brand_summary_aggs)
    .reset_index()
    .rename(columns={"brand_clean": "brand"})
)

In [ ]:
# Add brand-level share and composition features using cleaned brand labels.
brand_summary["brand_share_of_market"] = brand_summary["brand_total_registered"] / total_m1

if "vehicle_age" in m1_df.columns:
    over_10y = (m1_df["vehicle_age"] > 10).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_share_over_10y"] = brand_summary["brand"].map(over_10y)

if "matkamittarilukema" in m1_df.columns:
    over_200k_km = (m1_df["matkamittarilukema"] > 200000).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_share_over_200k_km"] = brand_summary["brand"].map(over_200k_km)

if "vaihteisto" in m1_df.columns:
    automatic_mask = m1_df["vaihteisto"].astype("string").isin(automatic_codes).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_automatic_share"] = brand_summary["brand"].map(automatic_mask)

if "kayttovoima" in m1_df.columns:
    powertrain = m1_df["kayttovoima"].astype("string")
    petrol_share = powertrain.isin(petrol_codes).groupby(m1_df["brand_clean"], dropna=False).mean()
    diesel_share = powertrain.isin(diesel_codes).groupby(m1_df["brand_clean"], dropna=False).mean()
    ev_share = powertrain.isin(ev_codes).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_petrol_share"] = brand_summary["brand"].map(petrol_share)
    brand_summary["brand_diesel_share"] = brand_summary["brand"].map(diesel_share)
    brand_summary["brand_ev_share"] = brand_summary["brand"].map(ev_share)

if "sahkohybridi" in m1_df.columns:
    hybrid_mask = m1_df["sahkohybridi"].astype("string").isin(true_values).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_hybrid_share"] = brand_summary["brand"].map(hybrid_mask)

if "ahdin" in m1_df.columns:
    turbo_mask = m1_df["ahdin"].astype("string").isin(true_values).groupby(m1_df["brand_clean"], dropna=False).mean()
    brand_summary["brand_turbo_share"] = brand_summary["brand"].map(turbo_mask)

brand_summary = brand_summary.sort_values("brand_total_registered", ascending=False).reset_index(drop=True)

In [ ]:
# Quick preview of the final cleaned brand_summary.
brand_summary.head()

,brand,brand_total_registered,brand_median_vehicle_age,brand_mean_vehicle_age,brand_median_mileage,brand_mean_mileage,brand_median_engine_cc,brand_median_power_kw,brand_median_mass_kg,brand_share_of_market,brand_share_over_10y,brand_share_over_200k_km,brand_automatic_share,brand_petrol_share,brand_diesel_share,brand_ev_share,brand_hybrid_share,brand_turbo_share
0,toyota,377068,13.0,13.210932,179335.5,193594.446161,1598.0,81.0,1345.0,0.147002,0.579262,0.381586,0.023903,0.925112,0.069295,0.004949,0.323491,0.083759
1,volkswagen,279311,11.0,12.154030,177745.0,192564.366137,1498.0,85.0,1400.0,0.108891,0.538110,0.386669,0.478356,0.656791,0.246449,0.074727,0.073083,0.686131
2,volvo,209869,14.0,14.073570,250631.5,256514.210612,1984.0,120.0,1665.0,0.081818,0.611034,0.584903,0.490511,0.490501,0.440055,0.058127,0.122786,0.709690
3,mercedes-benz,189399,11.0,12.570625,203892.5,226866.982380,2000.0,125.0,1700.0,0.073838,0.528250,0.470874,0.744671,0.403851,0.532564,0.060285,0.143438,0.806271
4,skoda,170359,8.0,9.062448,162500.0,179263.662746,1498.0,96.0,1375.0,0.066415,0.372549,0.311425,0.591551,0.630457,0.269020,0.066941,0.082039,0.830857


## Build `model_summary`
Aggregate the cleaned M1 market to model-family level using the cleaned brand and model keys.

In [ ]:
# Compute cleaned brand totals for model_share_within_brand.
brand_totals = (
    m1_df.groupby("brand_clean", dropna=False)
    .size()
    .rename("brand_total_registered")
)

In [ ]:
# Build the final model-level summary using cleaned grouping keys.
model_summary_aggs = {
    "model_total_registered": ("brand_clean", "size"),
}

if "vehicle_age" in m1_df.columns:
    model_summary_aggs["model_median_vehicle_age"] = ("vehicle_age", "median")
    model_summary_aggs["model_mean_vehicle_age"] = ("vehicle_age", "mean")

if "matkamittarilukema" in m1_df.columns:
    model_summary_aggs["model_median_mileage"] = ("matkamittarilukema", "median")
    model_summary_aggs["model_mean_mileage"] = ("matkamittarilukema", "mean")

if "iskutilavuus" in m1_df.columns:
    model_summary_aggs["model_median_engine_cc"] = ("iskutilavuus", "median")

if "suurinNettoteho" in m1_df.columns:
    model_summary_aggs["model_median_power_kw"] = ("suurinNettoteho", "median")

if "omamassa" in m1_df.columns:
    model_summary_aggs["model_median_mass_kg"] = ("omamassa", "median")

model_summary = (
    m1_df.groupby(["brand_clean", "model_family_clean"], dropna=False)
    .agg(**model_summary_aggs)
    .reset_index()
    .rename(columns={"brand_clean": "brand"})
)

In [ ]:
# Add model-level share and composition features using cleaned grouping keys.
model_summary["model_share_of_market"] = model_summary["model_total_registered"] / total_m1
model_summary["model_share_within_brand"] = (
    model_summary["model_total_registered"]
    / model_summary["brand"].map(brand_totals)
)

model_group = [m1_df["brand_clean"], m1_df["model_family_clean"]]
model_index = model_summary.set_index(["brand", "model_family_clean"]).index

if "vehicle_age" in m1_df.columns:
    over_10y = (m1_df["vehicle_age"] > 10).groupby(model_group, dropna=False).mean()
    model_summary["model_share_over_10y"] = model_index.map(over_10y)

if "matkamittarilukema" in m1_df.columns:
    over_200k_km = (m1_df["matkamittarilukema"] > 200000).groupby(model_group, dropna=False).mean()
    model_summary["model_share_over_200k_km"] = model_index.map(over_200k_km)

if "vaihteisto" in m1_df.columns:
    automatic_mask = m1_df["vaihteisto"].astype("string").isin(automatic_codes).groupby(model_group, dropna=False).mean()
    model_summary["model_automatic_share"] = model_index.map(automatic_mask)

if "kayttovoima" in m1_df.columns:
    powertrain = m1_df["kayttovoima"].astype("string")
    petrol_share = powertrain.isin(petrol_codes).groupby(model_group, dropna=False).mean()
    diesel_share = powertrain.isin(diesel_codes).groupby(model_group, dropna=False).mean()
    ev_share = powertrain.isin(ev_codes).groupby(model_group, dropna=False).mean()
    model_summary["model_petrol_share"] = model_index.map(petrol_share)
    model_summary["model_diesel_share"] = model_index.map(diesel_share)
    model_summary["model_ev_share"] = model_index.map(ev_share)

if "sahkohybridi" in m1_df.columns:
    hybrid_mask = m1_df["sahkohybridi"].astype("string").isin(true_values).groupby(model_group, dropna=False).mean()
    model_summary["model_hybrid_share"] = model_index.map(hybrid_mask)

if "ahdin" in m1_df.columns:
    turbo_mask = m1_df["ahdin"].astype("string").isin(true_values).groupby(model_group, dropna=False).mean()
    model_summary["model_turbo_share"] = model_index.map(turbo_mask)

model_summary = model_summary.sort_values(
    ["model_total_registered", "brand", "model_family_clean"],
    ascending=[False, True, True],
).reset_index(drop=True)

## Additional model lifecycle features from first-registration history
This section uses historical first-registration data by model series to enrich the Traficom model summary with lifecycle-oriented features. It was added later to describe the historical registration profile of the target models used in the thesis dataset.

In [ ]:
# Load the new full-market first-registration extracts used for lifecycle features.
project_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "datasets" / "traficom_datasets").exists()
)

brand_firstreg_path = project_root / "datasets" / "traficom_datasets" / "ensirek_20260410-141712.csv"
model_firstreg_path = project_root / "datasets" / "traficom_datasets" / "ensirek_20260410-142150.csv"


def read_firstreg_csv(path: Path) -> tuple[pd.DataFrame, str]:
    last_error = None
    for encoding in ["utf-8-sig", "utf-8", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=encoding, on_bad_lines="warn"), encoding
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


brand_firstreg_raw, brand_firstreg_encoding = read_firstreg_csv(brand_firstreg_path)
model_firstreg_raw, model_firstreg_encoding = read_firstreg_csv(model_firstreg_path)

print(f"Loaded brand first-registration file with encoding={brand_firstreg_encoding}: {brand_firstreg_path.name}")
print(f"Loaded model first-registration file with encoding={model_firstreg_encoding}: {model_firstreg_path.name}")
print("brand_firstreg_raw shape:", brand_firstreg_raw.shape)
print("model_firstreg_raw shape:", model_firstreg_raw.shape)
print("brand_firstreg_raw columns:", brand_firstreg_raw.columns.tolist())
print("model_firstreg_raw columns:", model_firstreg_raw.columns.tolist())


Loaded lifecycle file with encoding=utf-8-sig
(78, 5)
['Alue', 'Mallisarja', 'K�ytt�voima', 'Vuosi', 'Henkil�autojen ensirekister�innit']


In [ ]:
# Clean the full-market registration extracts and rebuild merge-ready keys.
import re

FIRSTREG_SUMMARY_END_YEAR = 2025
FIRSTREG_RECENT_START_YEAR = 2021
FIRSTREG_OLD_END_YEAR = 2018
TOTAL_ROW_PATTERN = r"yhteensä$"
brand_alias_map = {
    "vw": "volkswagen",
    "volkswagen, vw": "volkswagen",
    "skd": "skoda",
    "skida": "skoda",
}


def standardize_key(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"": pd.NA, "nan": pd.NA})
    )


brand_prefix_aliases = {alias: canonical for alias, canonical in brand_alias_map.items() if alias != canonical}
known_brands = sorted(m1_df["brand_clean"].dropna().unique().tolist(), key=len, reverse=True)
brand_prefix_lookup = {brand: brand for brand in known_brands}
brand_prefix_lookup.update(brand_prefix_aliases)


def brand_to_pattern(label: str) -> str:
    return re.escape(label).replace(r"\ ", r"[- ]+").replace(r"\-", r"[- ]+")


brand_prefix_patterns = [
    (alias, canonical, re.compile(rf"^{brand_to_pattern(alias)}(?:[- ]+|$)"))
    for alias, canonical in sorted(brand_prefix_lookup.items(), key=lambda item: len(item[0]), reverse=True)
]


def parse_model_registration_key(label: object) -> tuple[object, object]:
    if pd.isna(label):
        return pd.NA, pd.NA

    cleaned_label = standardize_key(pd.Series([label])).iloc[0]
    if pd.isna(cleaned_label):
        return pd.NA, pd.NA

    for alias, canonical, pattern in brand_prefix_patterns:
        if pattern.match(cleaned_label):
            model_key = pattern.sub("", cleaned_label, count=1).strip(" -")
            model_key = re.sub(r"\s+", " ", model_key)
            return canonical, (model_key if model_key else pd.NA)

    return pd.NA, cleaned_label


def clean_registration_counts(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype("string")
        .str.replace(r"\s+", "", regex=True)
        .str.replace(",", ".", regex=False)
        .replace({"-": pd.NA}),
        errors="coerce",
    )


brand_firstreg = brand_firstreg_raw.rename(
    columns={
        "Maakunta": "region",
        "Merkki": "brand_label",
        "Käyttövoima": "powertrain",
        "Kuukausi": "month_scope",
        "Vuosi": "vuosi",
        "Henkilöautojen ensirekisteröinnit": "first_registrations",
    }
).copy()

for col in ["region", "brand_label", "powertrain", "month_scope"]:
    brand_firstreg[col] = brand_firstreg[col].astype("string").str.strip()

brand_firstreg["brand"] = standardize_key(brand_firstreg["brand_label"]).replace(brand_alias_map)
brand_firstreg["vuosi"] = pd.to_numeric(brand_firstreg["vuosi"], errors="coerce")
brand_firstreg["first_registrations"] = clean_registration_counts(brand_firstreg["first_registrations"])
brand_firstreg = brand_firstreg[
    brand_firstreg["brand_label"].astype("string").str.contains(TOTAL_ROW_PATTERN, case=False, na=False) == False
].dropna(subset=["brand", "vuosi", "first_registrations"]).copy()
brand_firstreg["vuosi"] = brand_firstreg["vuosi"].astype(int)
brand_firstreg = brand_firstreg[brand_firstreg["vuosi"].between(2016, 2026)].copy()

model_firstreg = model_firstreg_raw.rename(
    columns={
        "Alue": "region",
        "Mallisarja": "mallisarja",
        "Käyttövoima": "powertrain",
        "Vuosi": "vuosi",
        "Henkilöautojen ensirekisteröinnit": "first_registrations",
    }
).copy()

for col in ["region", "mallisarja", "powertrain"]:
    model_firstreg[col] = model_firstreg[col].astype("string").str.strip()

model_firstreg[["brand", "model_family_clean"]] = pd.DataFrame(
    model_firstreg["mallisarja"].map(parse_model_registration_key).tolist(),
    index=model_firstreg.index,
)
model_firstreg["vuosi"] = pd.to_numeric(model_firstreg["vuosi"], errors="coerce")
model_firstreg["first_registrations"] = clean_registration_counts(model_firstreg["first_registrations"])
model_firstreg = model_firstreg[
    model_firstreg["mallisarja"].astype("string").str.contains(TOTAL_ROW_PATTERN, case=False, na=False) == False
].dropna(subset=["brand", "model_family_clean", "vuosi", "first_registrations"]).copy()
model_firstreg["vuosi"] = model_firstreg["vuosi"].astype(int)
model_firstreg = model_firstreg[model_firstreg["vuosi"].between(2014, 2026)].copy()

print("Cleaned brand first-registration shape:", brand_firstreg.shape)
print("Cleaned model first-registration shape:", model_firstreg.shape)
print("Unique cleaned brand keys in brand first-registration data:", brand_firstreg["brand"].nunique(dropna=True))
print("Unique cleaned brand keys in model first-registration data:", model_firstreg["brand"].nunique(dropna=True))
print("Unique cleaned model keys in model first-registration data:", model_firstreg["model_family_clean"].nunique(dropna=True))
print("Brand first-registration years kept:", sorted(brand_firstreg["vuosi"].unique().tolist()))
print("Model first-registration years kept:", sorted(model_firstreg["vuosi"].unique().tolist()))
print("Rows with unresolved model-registration brand keys:", int(model_firstreg_raw.shape[0] - model_firstreg.shape[0]))


(39, 5)
mallisarja
SKODA OCTAVIA      13
TOYOTA COROLLA     13
VOLKSWAGEN GOLF    13
Name: count, dtype: Int64


In [ ]:
# Aggregate lifecycle features from the cleaned full-market registration tables.
# 2026 is intentionally excluded from the summary calculations because the downloaded files are partial for that year.
# The existing *_2014_2026 column names are kept for downstream notebook compatibility.

def summarize_first_registration(df: pd.DataFrame, group_cols: list[str], prefix: str) -> pd.DataFrame:
    scope_df = df[df["vuosi"].between(int(df["vuosi"].min()), FIRSTREG_SUMMARY_END_YEAR)].copy()
    summary_rows = []

    for group_key, group in scope_df.groupby(group_cols, dropna=False):
        key_values = group_key if isinstance(group_key, tuple) else (group_key,)
        row = dict(zip(group_cols, key_values))

        total = group["first_registrations"].sum()
        recent_total = group.loc[group["vuosi"].between(FIRSTREG_RECENT_START_YEAR, FIRSTREG_SUMMARY_END_YEAR), "first_registrations"].sum()
        old_total = group.loc[group["vuosi"] <= FIRSTREG_OLD_END_YEAR, "first_registrations"].sum()
        nonzero_years = group.loc[group["first_registrations"] > 0, "vuosi"]
        peak_row = group.sort_values(["first_registrations", "vuosi"], ascending=[False, True]).iloc[0]

        row.update(
            {
                f"{prefix}_firstreg_total_2014_2026": total,
                f"{prefix}_firstreg_recent_share": recent_total / total if total > 0 else pd.NA,
                f"{prefix}_firstreg_old_share": old_total / total if total > 0 else pd.NA,
                f"{prefix}_firstreg_weighted_year": (
                    (group["vuosi"] * group["first_registrations"]).sum() / total if total > 0 else pd.NA
                ),
                f"{prefix}_firstreg_peak_year": int(peak_row["vuosi"]) if total > 0 else pd.NA,
                f"{prefix}_firstreg_peak_count": peak_row["first_registrations"] if total > 0 else pd.NA,
                f"{prefix}_firstreg_year_span": (
                    int(nonzero_years.max() - nonzero_years.min()) if not nonzero_years.empty else pd.NA
                ),
            }
        )
        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


brand_firstreg_summary = summarize_first_registration(brand_firstreg, ["brand"], "brand")
model_firstreg_summary = summarize_first_registration(model_firstreg, ["brand", "model_family_clean"], "model")

print("brand_firstreg_summary shape:", brand_firstreg_summary.shape)
print("model_firstreg_summary shape:", model_firstreg_summary.shape)
print("Sample brand first-registration summary rows:")
print(brand_firstreg_summary.head(10).to_string(index=False))
print("\nSample model first-registration summary rows:")
print(model_firstreg_summary.head(10).to_string(index=False))


      mallisarja  brand model_family_clean  vuosi  first_registrations
0  SKODA OCTAVIA  skoda            octavia   2014                 5868
1  SKODA OCTAVIA  skoda            octavia   2015                 5833
2  SKODA OCTAVIA  skoda            octavia   2016                 5532
3  SKODA OCTAVIA  skoda            octavia   2017                 5694
4  SKODA OCTAVIA  skoda            octavia   2018                 5000


In [ ]:
# Merge brand-level first-registration summaries into the existing stock-style brand summary.
brand_summary.columns = brand_summary.columns.str.strip()
brand_firstreg_summary.columns = brand_firstreg_summary.columns.str.strip()
brand_summary["brand"] = standardize_key(brand_summary["brand"])
brand_firstreg_summary["brand"] = standardize_key(brand_firstreg_summary["brand"])

brand_summary_rows_before = len(brand_summary)
brand_firstreg_feature_cols = [col for col in brand_firstreg_summary.columns if col != "brand"]

brand_summary_keyed = brand_summary.dropna(subset=["brand"]).copy()
brand_summary_missing_key = brand_summary[brand_summary["brand"].isna()].copy()
brand_firstreg_keyed = brand_firstreg_summary.dropna(subset=["brand"]).copy()

print(f"Duplicate keyed brand rows in stock summary: {int(brand_summary_keyed[['brand']].duplicated().sum())}")
print(f"Duplicate keyed brand rows in first-registration summary: {int(brand_firstreg_keyed[['brand']].duplicated().sum())}")
brand_summary_keyed = brand_summary_keyed.merge(
    brand_firstreg_keyed,
    on="brand",
    how="left",
    validate="many_to_one",
)
for col in brand_firstreg_feature_cols:
    brand_summary_missing_key[col] = pd.NA

brand_summary = pd.concat([brand_summary_keyed, brand_summary_missing_key], ignore_index=True)
brand_summary_rows_after = len(brand_summary)

brand_firstreg_match_count = int(brand_summary["brand_firstreg_total_2014_2026"].notna().sum())
brand_firstreg_missing_count = int(brand_summary["brand_firstreg_total_2014_2026"].isna().sum())

print(f"brand_summary rows before first-registration merge: {brand_summary_rows_before}")
print(f"brand_summary rows after first-registration merge: {brand_summary_rows_after}")
print(f"Brand rows with missing stock keys skipped from validated merge: {len(brand_summary_missing_key)}")
print(f"Brand rows matched to first-registration summaries: {brand_firstreg_match_count}")
print(f"Brand rows missing first-registration summaries: {brand_firstreg_missing_count}")
print("Sample merged brand lifecycle features:")
print(
    brand_summary[
        [
            col for col in [
                "brand",
                "brand_total_registered",
                "brand_firstreg_total_2014_2026",
                "brand_firstreg_recent_share",
                "brand_firstreg_old_share",
                "brand_firstreg_weighted_year",
                "brand_firstreg_peak_year",
            ] if col in brand_summary.columns
        ]
    ].head(10).to_string(index=False)
)


model lifecycle feature shape: (3, 9)
brand lifecycle feature shape: (3, 8)
brand_summary rows before lifecycle merge: 768
brand_summary rows after lifecycle merge: 768


(        brand model_family_clean  model_firstreg_total_2014_2026  \
 0       skoda            octavia                           48859   
 1      toyota            corolla                           35135   
 2  volkswagen               golf                           31860   
 
    model_firstreg_recent_share  model_firstreg_old_share  \
 0                     0.234041                  0.571584   
 1                     0.601793                  0.102234   
 2                     0.169586                  0.668707   
 
    model_firstreg_weighted_year  model_firstreg_peak_year  \
 0                   2018.163225                      2014   
 1                   2021.013434                      2020   
 2                   2017.604237                      2014   
 
    model_firstreg_peak_count  model_firstreg_year_span  
 0                       5868                        12  
 1                       5395                        12  
 2                       5265                       

In [ ]:
# Merge model-level first-registration summaries into the existing stock-style model summary.
model_summary.columns = model_summary.columns.str.strip()
model_firstreg_summary.columns = model_firstreg_summary.columns.str.strip()

for _df in [model_summary, model_firstreg_summary]:
    _df["brand"] = standardize_key(_df["brand"])
    _df["model_family_clean"] = standardize_key(_df["model_family_clean"])

model_summary_rows_before = len(model_summary)
model_firstreg_feature_cols = [col for col in model_firstreg_summary.columns if col not in ["brand", "model_family_clean"]]

model_summary_keyed = model_summary.dropna(subset=["brand", "model_family_clean"]).copy()
model_summary_missing_key = model_summary[
    model_summary[['brand', 'model_family_clean']].isna().any(axis=1)
].copy()
model_firstreg_keyed = model_firstreg_summary.dropna(subset=["brand", "model_family_clean"]).copy()

print(f"Duplicate keyed model rows in stock summary: {int(model_summary_keyed[['brand', 'model_family_clean']].duplicated().sum())}")
print(f"Duplicate keyed model rows in first-registration summary: {int(model_firstreg_keyed[['brand', 'model_family_clean']].duplicated().sum())}")
model_summary_keyed = model_summary_keyed.merge(
    model_firstreg_keyed,
    on=["brand", "model_family_clean"],
    how="left",
    validate="many_to_one",
)
for col in model_firstreg_feature_cols:
    model_summary_missing_key[col] = pd.NA

model_summary = pd.concat([model_summary_keyed, model_summary_missing_key], ignore_index=True)
model_summary_rows_after = len(model_summary)

model_firstreg_match_count = int(model_summary["model_firstreg_total_2014_2026"].notna().sum())
model_firstreg_missing_count = int(model_summary["model_firstreg_total_2014_2026"].isna().sum())

print(f"model_summary rows before merge: {model_summary_rows_before}")
print(f"model_summary rows after merge: {model_summary_rows_after}")
print(f"Model rows with missing stock keys skipped from validated merge: {len(model_summary_missing_key)}")
print(f"Model rows matched to first-registration summaries: {model_firstreg_match_count}")
print(f"Model rows missing first-registration summaries: {model_firstreg_missing_count}")


model_summary rows before merge: 19054
model_summary rows after merge: 19054
rows matched to lifecycle features: 3
rows missing lifecycle features after merge: 19051


In [ ]:
# Validate first-registration coverage against the existing stock-summary keyspace.
firstreg_feature_cols = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
]
brand_firstreg_feature_cols = [
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

missing_lifecycle_cols = [col for col in firstreg_feature_cols if col not in model_summary.columns]
missing_brand_lifecycle_cols = [col for col in brand_firstreg_feature_cols if col not in brand_summary.columns]

model_unmatched_keys = (
    model_summary.loc[
        model_summary["model_firstreg_total_2014_2026"].isna(),
        ["brand", "model_family_clean"],
    ]
    .dropna()
    .drop_duplicates()
    .head(20)
)
brand_unmatched_keys = (
    brand_summary.loc[
        brand_summary["brand_firstreg_total_2014_2026"].isna(),
        ["brand"],
    ]
    .dropna()
    .drop_duplicates()
    .head(20)
)

print(f"Missing model lifecycle columns: {missing_lifecycle_cols}")
print(f"Missing brand lifecycle columns: {missing_brand_lifecycle_cols}")
print("\nTop unmatched model keys after first-registration merge:")
print(model_unmatched_keys.to_string(index=False))
print("\nTop unmatched brand keys after first-registration merge:")
print(brand_unmatched_keys.to_string(index=False))

registration_target_models = model_summary[
    model_summary["model_family_clean"].isin(["corolla", "octavia", "golf"])
][["brand", "model_family_clean"] + firstreg_feature_cols].head(20)
print("\nSample target-model lifecycle rows:")
print(registration_target_models.to_string(index=False))


Missing lifecycle columns: []
Target model lifecycle rows populated:
        brand model_family_clean  has_lifecycle
0  volkswagen               golf           True
1       skoda            octavia           True
2      toyota            corolla           True


(        brand model_family_clean  model_firstreg_total_2014_2026  \
 0  volkswagen               golf                         31860.0   
 1       skoda            octavia                         48859.0   
 2      toyota            corolla                         35135.0   
 
    model_firstreg_recent_share  model_firstreg_old_share  \
 0                     0.169586                  0.668707   
 1                     0.234041                  0.571584   
 2                     0.601793                  0.102234   
 
    model_firstreg_weighted_year  model_firstreg_peak_year  \
 0                   2017.604237                    2014.0   
 1                   2018.163225                    2014.0   
 2                   2021.013434                    2020.0   
 
    model_firstreg_peak_count  model_firstreg_year_span  
 0                     5265.0                      12.0  
 1                     5868.0                      12.0  
 2                     5395.0                      1

## Inspect target models
Check that the final cleaned summary tables collapse the target brands and models into consistent labels.

In [ ]:
# Preview the final summary tables and target model families.
target_models = model_summary[
    ((model_summary["brand"] == "toyota") & (model_summary["model_family_clean"] == "corolla"))
    | ((model_summary["brand"] == "skoda") & (model_summary["model_family_clean"] == "octavia"))
    | ((model_summary["brand"] == "volkswagen") & (model_summary["model_family_clean"] == "golf"))
]

model_summary.head(), target_models.head(20)

(        brand model_family_clean  model_total_registered  \
 0  volkswagen               golf                   86762   
 1       skoda            octavia                   82555   
 2      toyota            corolla                   59454   
 3        ford              focus                   57063   
 4  volkswagen             passat                   50893   
 
    model_median_vehicle_age  model_mean_vehicle_age  model_median_mileage  \
 0                      13.0               13.748634              186801.0   
 1                      10.0               10.601824              190191.0   
 2                      21.0               22.448058              249570.5   
 3                      14.0               13.417153              181418.5   
 4                      11.0               12.074060              223236.0   
 
    model_mean_mileage  model_median_engine_cc  model_median_power_kw  \
 0       195467.967875                  1395.0                   81.0   
 1       201919.

## Export final summary tables
Write only the final reusable Traficom summary outputs for later merging work.

In [ ]:
# Confirm the final exported objects carry the rebuilt lifecycle features.
model_summary.columns = model_summary.columns.str.strip()
brand_summary.columns = brand_summary.columns.str.strip()

model_lifecycle_export_cols = [c for c in model_summary.columns if c.startswith("model_firstreg_")]
brand_lifecycle_export_cols = [c for c in brand_summary.columns if c.startswith("brand_firstreg_")]

print("model_summary shape:", model_summary.shape)
print("brand_summary shape:", brand_summary.shape)
print("model lifecycle export columns:", model_lifecycle_export_cols)
print("brand lifecycle export columns:", brand_lifecycle_export_cols)
print("\nmodel_summary preview:")
print(model_summary[["brand", "model_family_clean"] + model_lifecycle_export_cols].head(10).to_string(index=False))
print("\nbrand_summary preview:")
print(brand_summary[["brand"] + brand_lifecycle_export_cols].head(10).to_string(index=False))


(19054, 27)
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span']


,brand,model_family_clean,model_firstreg_total_2014_2026,model_firstreg_recent_share,model_firstreg_old_share,model_firstreg_weighted_year,model_firstreg_peak_year,model_firstreg_peak_count,model_firstreg_year_span
0,volkswagen,golf,31860.0,0.169586,0.668707,2017.604237,2014.0,5265.0,12.0
1,skoda,octavia,48859.0,0.234041,0.571584,2018.163225,2014.0,5868.0,12.0
2,toyota,corolla,35135.0,0.601793,0.102234,2021.013434,2020.0,5395.0,12.0


In [ ]:
output_dir = project_root / "datasets" / "traficom_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

brand_summary.columns = brand_summary.columns.str.strip()
model_summary.columns = model_summary.columns.str.strip()
brand_firstreg_summary.columns = brand_firstreg_summary.columns.str.strip()
model_firstreg_summary.columns = model_firstreg_summary.columns.str.strip()

brand_firstreg_summary.to_csv(output_dir / "brand_firstreg_summary.csv", index=False)
model_firstreg_summary.to_csv(output_dir / "model_firstreg_summary.csv", index=False)
brand_summary.to_csv(output_dir / "brand_summary.csv", index=False)
model_summary.to_csv(output_dir / "model_summary.csv", index=False)

print(output_dir / "brand_firstreg_summary.csv")
print(output_dir / "model_firstreg_summary.csv")
print(output_dir / "brand_summary.csv")
print(output_dir / "model_summary.csv")


/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/traficom_outputs/brand_summary.csv
/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/traficom_outputs/model_summary.csv
